# Build a sales agent in Colab

Run this `.ipynb` file in [Google Colab](https://colab.research.google.com/). Everything below runs from cells; no terminal or local Python installation is needed.

Start with section 0, then run the free exercises in order. Paid runs are clearly marked and switched off by default. Set a paid cell's flag to `True` only when you want to run it, then change it back to `False` afterward.

Your submission is still **`prompt.py` and `tools.py` plus five saved runs**. The notebook helps you develop and run those files; it is not the submission.

## 0. Set up in Colab

1. Unzip `sales-agent-homework.zip` on your computer to find `explore.ipynb`. Keep the original ZIP too.
2. Open [Google Colab](https://colab.research.google.com/), choose **File → Upload notebook**, and select `explore.ipynb`. Save a copy to your Google Drive to keep your cell edits.
3. Connect to a Python runtime. The default CPU runtime is enough; no GPU is needed.
4. Open **Files** (folder icon on the left). Upload the original **`sales-agent-homework.zip`** into the top-level `/content` folder. Wait for the upload to finish.
5. Click the play button beside the setup cell below, then beside the check cell. Wait for each to finish before continuing. The first installation can take a few minutes. Continue when you see **“Setup checks passed”** and **“OK”** after the tests.

No API key is needed for setup or the free exercises in sections 1–5.

Rerunning setup keeps your existing homework files. If Colab asks you to restart after installation, restart and run both cells again. See [Colab's help](https://research.google.com/colaboratory/faq.html) for opening files and managing sessions.

### Resume from a backup

If your homework folder is still in the Files panel, rerun the two setup cells to continue working. If it is gone:

1. Upload the backup ZIP you saved in section 11 to the top level of the **Files** panel.
2. Change `ARCHIVE_NAME` in the cell below to the backup's exact filename.
3. Run the setup cell, then the check cell.

This restores your code, run results and saved model responses. To restore a backup instead of using an existing homework folder, start a fresh Colab session first; setup will not overwrite that folder.


In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path, PurePosixPath

ARCHIVE_NAME = "sales-agent-homework.zip"  # Or the ZIP saved by section 11.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
UPLOAD_DIR = Path("/content") if IN_COLAB else Path.cwd()
WORKSPACE = UPLOAD_DIR / "sales-agent-homework"
REQUIRED_FILES = ("run.py", "prompt.py", "tools.py", "requirements.txt")

def is_homework(folder):
    return all((folder / name).is_file() for name in REQUIRED_FILES)

candidates = [Path.cwd(), WORKSPACE]
if WORKSPACE.is_dir():
    candidates += [p.parent for p in WORKSPACE.rglob("requirements.txt")]
ROOT = next((p.resolve() for p in candidates if is_homework(p)), None)

if ROOT is None:
    archive = UPLOAD_DIR / ARCHIVE_NAME
    if not archive.is_file():
        raise FileNotFoundError(
            f"Upload {ARCHIVE_NAME} to {UPLOAD_DIR} using Colab's Files panel, then rerun this cell."
        )
    if WORKSPACE.exists() and any(WORKSPACE.iterdir()):
        raise RuntimeError(
            f"{WORKSPACE} contains an incomplete homework folder. Rename it in Files before "
            "rerunning setup; your existing files will not be overwritten."
        )
    with zipfile.ZipFile(archive) as bundle:
        for entry in bundle.infolist():
            relative = PurePosixPath(entry.filename)
            if relative.is_absolute() or ".." in relative.parts:
                raise ValueError("This ZIP contains an invalid file path. Use the supplied homework or a backup.")
        bundle.extractall(WORKSPACE)
    candidates = [WORKSPACE] + [p.parent for p in WORKSPACE.rglob("requirements.txt")]
    found = [p.resolve() for p in candidates if is_homework(p)]
    found = list(dict.fromkeys(found))
    if len(found) != 1:
        raise RuntimeError("Could not identify one homework folder in the ZIP. Check the uploaded file.")
    ROOT = found[0]

os.chdir(ROOT)
print("Homework folder:", ROOT)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=ROOT, check=True,
)
print("Installation finished. Run the next cell to check the setup.")

In [ ]:
from sales_sim import tool
from sales_sim.notebook import list_cases, run_case, show_prompt, show_result, try_tool, try_tools
from sales_sim.student_setup import check_setup, model_settings
from sales_sim.experiments import default_output

if not check_setup(ROOT)["ok"]:
    raise RuntimeError("Fix the setup errors above. If installation requested a restart, restart the session and rerun section 0.")
subprocess.run([sys.executable, "run.py", "test"], cwd=ROOT, check=True)
settings = model_settings(ROOT)
CAMPAIGN = "full_campaign"
LAST_OUTPUT = None
FINAL_OUTPUT = None

## 1. What the model reads

The first model call receives your prompt, the note that opens day 1 and the full tool definitions. Later calls also include the conversation and tool results so far.


In [ ]:
seen = show_prompt(root=ROOT)

## 2. Call a tool the way the model does

`try_tool` runs one call on a fresh copy of the 40-account campaign, exactly as a run does. `output` is the text the model reads next. Reading is free. Sending a customer a message is not possible here, because these customers are simulated by a paid model (section 4 shows messages on a free practice case).

`variant="baseline"` uses the 14 supplied tools. Leave it out to use your saved `tools.py`.

In [ ]:
accounts = try_tool("list_customers", case=CAMPAIGN, root=ROOT, variant="baseline")
print(len(accounts["result"]), "accounts")
show_result(accounts)

In [ ]:
customer_id = accounts["result"][0]["id"]
history = try_tool("read_customer_history", {"customer_id": customer_id}, case=CAMPAIGN, root=ROOT, variant="baseline")
show_result(history)

The company's documents describe its products and policies, including what changes during the campaign:

In [ ]:
documents = try_tool("list_documents", case=CAMPAIGN, root=ROOT, variant="baseline")
for document in documents["result"]:
    print(f'{document["id"]:30} {document["title"]}')

Errors come back to the model as text, and the run goes on:

In [ ]:
missing = try_tool("read_customer_history", {"customer_id": "nobody"}, case=CAMPAIGN, root=ROOT, variant="baseline")
wrong_type = try_tool("read_customer_history", {"customer_id": 3}, case=CAMPAIGN, root=ROOT, variant="baseline")
print(missing["output"])
print(wrong_type["output"])

## 3. Write a tool

A tool is a Python function marked `@tool`. The model sees its name, its docstring and its arguments. `ctx` gives it the company's operations, listed with real outputs in `OPERATIONS.md`. Try a tool here with `tools=[...]`, then move it into `tools.py`.

This one does arithmetic the model often gets wrong: the range of prices it may quote today.

In [ ]:
@tool
def price_range(ctx, product_id: str) -> dict:
    """The lowest and highest monthly price allowed today for one product, in cents."""
    catalog = ctx.get_catalog()
    policy = catalog["current_pricing_policy"]
    for product in catalog["products"]:
        if product["id"] == product_id:
            list_price = product["monthly_price_cents"]
            discounted = -(-list_price * (100 - policy["max_discount_percent"]) // 100)  # rounded up
            return {"lowest_cents": max(discounted, product["monthly_service_cost_cents"]),
                    "highest_cents": list_price}
    return {"error": f"No product {product_id!r}."}


prices = try_tool("price_range", {"product_id": "pro"}, case=CAMPAIGN, root=ROOT, tools=[price_range])
show_result(prices)

Every tool output stays in the conversation and is sent again with every later model call, so size matters. `list_customers` returns about 4,000 tokens. A tool can return only what the model needs:

In [ ]:
@tool
def account_table(ctx) -> list:
    """One line per account: ID | business | status | last contact day | CRM background excerpt."""
    lines = []
    for customer in ctx.list_customers():
        latest = customer["background"].split("Latest contact: ")[-1][:70]
        last = customer["contact"]["last_contact_day"] or "-"
        lines.append(f'{customer["id"]} | {customer["business"]} | {customer["status"]} | day {last} | {latest}')
    return lines


table = try_tool("account_table", case=CAMPAIGN, root=ROOT, tools=[account_table])
print(f'list_customers: about {accounts["calls"][0]["approx_tokens"]:,} tokens; '
      f'account_table: about {table["calls"][0]["approx_tokens"]:,} tokens')
show_result(table)

A bug in a tool doesn't stop the run: the model is told the tool crashed, with the file and line.

In [ ]:
@tool
def buggy_brief(ctx, customer_id: str) -> dict:
    """A deliberately broken tool."""
    return ctx.read_customer(customer_id=customer_id)["no_such_field"]


crash = try_tool("buggy_brief", {"customer_id": customer_id}, case=CAMPAIGN, root=ROOT, tools=[buggy_brief])
print(crash["output"])

## 4. Messages, quotes and days, on a practice case

Each practice case has one scripted customer, so messages are free here. `try_tools` runs a sequence of calls in one campaign; `"end_day"` ends the day. In `issued_quote` prices rise on day 2 and the customer decides on day 3: a quote presented on day 1 keeps its price.

In [ ]:
for case in list_cases(ROOT):
    print(f'{case["id"]:24} {case["description"]}')

In [ ]:
sequence = try_tools([
    ("create_quote", {"customer_id": "orion", "product_id": "pro", "monthly_price_cents": 5500}),
    ("present_quote", {"quote_id": "run:quote:Q0001", "message": "Pro is $55.00 per month."}),
    "end_day",
    "end_day",
    ("present_quote", {"quote_id": "run:quote:Q0001", "message": "Your $55.00 quote still stands. Shall we go ahead?"}),
    "end_day",
], case="issued_quote", root=ROOT, variant="baseline")
show_result(sequence)
print("Profit:", sequence["outcome"]["profit_cents"] / 100)

## 5. A whole run, offline

The real loop with a script in place of the model, on a practice case. It saves the same files as a paid run. The report appears below; section 9 also lets you download it and open it in your browser.


In [ ]:
demonstration = run_case("requirements_update", reference=True, root=ROOT)
LAST_OUTPUT = Path(demonstration["output"])
show_result(demonstration)


## 6. Edit your submitted files

1. Open **Files** (folder icon on the left) and expand the homework folder shown in section 0.
2. Double-click **`prompt.py`** to edit `SYSTEM_PROMPT`, or **`tools.py`** to edit your tools.
3. Click inside the file editor and save with **Ctrl+S** (Windows/Linux) or **⌘S** (Mac).
4. Run the cell below to check your files and preview what the agent will read.

If you wrote a tool in an exercise cell, copy it into `tools.py` and save the file so your agent can use it. Keep any basic tools your agent still needs. Each run uses your saved files.

Before closing Colab, follow **section 11** to save your work.


In [ ]:
subprocess.run([sys.executable, "run.py", "test"], cwd=ROOT, check=True)
seen = show_prompt(root=ROOT)

## 7. Connect Parley and try a paid practice run

Paid calls use MIT Parley. Graduate students get $30 of API credit a month. A baseline 40-account campaign costs about $0.55; your version may use more.

### Connect your API key

1. Sign in at [parley.mit.edu](https://parley.mit.edu) with your MIT credentials and choose **Parley API**, or go directly to [platform.parley.mit.edu](https://platform.parley.mit.edu). Generate a new key and copy it.
2. In Colab, open **Secrets** (key icon on the left). Add **`PARLEY_API_KEY`**, paste the key as its value and enable notebook access.
3. Run the helper cell below. It prepares the commands used by later cells. It is free and does not read your key until you start a paid run.

Treat your key like a password. Never paste it into a code cell or source file.

### Costs and spending

The campaign's contact and service costs are simulated. Model calls charge your Parley account. Longer prompts and tool outputs can increase the API bill.

`BUDGET_USD = 20` sets an approximate spending cap for this homework. It uses the saved usage record, not your Parley account balance. Keep one paid run active at a time. All paid runs use the fixed course model settings.

Use **Check your spending** in section 9 to see the total. Save a backup in section 11 to keep the usage record and replay earlier runs for free.


In [ ]:
import json
import math
from getpass import getpass
from html import escape
from IPython.display import HTML, display

BUDGET_USD = 20

def prepare_paid():
    if not math.isfinite(BUDGET_USD) or BUDGET_USD <= 0:
        raise ValueError("BUDGET_USD must be a positive number.")
    os.environ["SALES_SIM_BUDGET_USD"] = str(BUDGET_USD)
    if not os.environ.get("PARLEY_API_KEY", "").strip():
        if IN_COLAB:
            from google.colab import userdata
            try:
                key = userdata.get("PARLEY_API_KEY")
            except Exception:
                raise RuntimeError("Add PARLEY_API_KEY in Colab Secrets and enable notebook access, then rerun this cell.") from None
        else:
            key = getpass("MIT Parley API key: ")
        if not key or not key.strip():
            raise RuntimeError("No MIT Parley API key was provided.")
        os.environ["PARLEY_API_KEY"] = key.strip()

def run_command(*arguments):
    subprocess.run([sys.executable, "run.py", *map(str, arguments)], cwd=ROOT, check=True)

def download_file(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    if IN_COLAB:
        from google.colab import files
        files.download(str(path))
    else:
        print("Saved file:", path)

def show_report(output, run_number=1):
    pages = sorted((Path(output) / "runs").glob("*/inspect.html"))
    if not 1 <= run_number <= len(pages):
        raise ValueError(f"Choose REPORT_RUN from 1 to {len(pages)}.")
    report = pages[run_number - 1]
    page = escape(report.read_text(encoding="utf-8"), quote=True)
    display(HTML(
        '<iframe title="Sales run report" sandbox="allow-scripts allow-downloads" '
        'style="width:100%;height:760px;border:1px solid #ddd" '
        f'srcdoc="{page}"></iframe>'
    ))
    print(f"Showing run {run_number} of {len(pages)}. If the report is blank, download it using the report cell below.")
    return report

In [ ]:
RUN_PRACTICE = False
PRACTICE_CASE = "requirements_update"

if RUN_PRACTICE:  # A few cents.
    prepare_paid()
    output = default_output(ROOT, "practice")
    run_command("run", "--case", PRACTICE_CASE, "--output", output)
    LAST_OUTPUT = output
    REPORT_FILE = show_report(LAST_OUTPUT)
else:
    print("Paid practice skipped. Set RUN_PRACTICE = True when you are ready.")

## 8. Run the starting agent

This runs the supplied agent on the **40-account, 12-day campaign** (about $0.55 and 5–10 minutes).

1. Set `RUN_BASELINE = True` and run the cell below.
2. Wait for the run to finish. Its interactive report appears below the cell.
3. Change `RUN_BASELINE` back to `False` so running all cells will not launch it again.

Use the report to find missed opportunities, incorrect offers, tool errors and promises that were not kept. If the report does not display, use **View or download a report** in section 9 to open it in your browser.


In [ ]:
RUN_BASELINE = False

if RUN_BASELINE:
    prepare_paid()
    output = default_output(ROOT, "baseline")
    run_command("run", "--suite", "development/full_campaign/suite.json",
                "--variant", "baseline", "--output", output)
    LAST_OUTPUT = output
    REPORT_FILE = show_report(LAST_OUTPUT)
else:
    print("Baseline skipped. Set RUN_BASELINE = True when you are ready.")

## 9. Run your improved agent and inspect results

1. Save `prompt.py` and `tools.py`, then rerun section 6's check cell.
2. Choose `REPEATS = 1` for one trial (about $1), or `REPEATS = 3` for three independent trials (about $3).
3. Set `RUN_CAMPAIGN = True` and run the cell below.
4. When it finishes, change `RUN_CAMPAIGN` back to `False`.

Unchanged runs replay saved responses for free while the response cache is available. Keep `FRESH = False` to use them. Set `FRESH = True` only when you want new samples and new API charges.

Compare profit and the conversations in the reports. Check whether the accounts your change targeted behaved differently.


In [ ]:
RUN_CAMPAIGN = False
REPEATS = 1
FRESH = False

if RUN_CAMPAIGN:
    prepare_paid()
    output = default_output(ROOT, "tools")
    arguments = ["run", "--suite", "development/full_campaign/suite.json",
                 "--repeats", REPEATS, "--output", output]
    if FRESH:
        arguments.append("--fresh")
    run_command(*arguments)
    LAST_OUTPUT = output
    REPORT_FILE = show_report(LAST_OUTPUT)
else:
    print("Campaign skipped. Set RUN_CAMPAIGN = True when you are ready.")

### View or download a report

The next cell opens a report from the last run in this session, including the free demonstration. Run section 7's helper cell first if you have not already; it is free and does not need an API key.

1. For a batch of runs, set `REPORT_RUN` to the run number you want: 1, 2, etc.
2. Run the cell to view the report.
3. To open it in your browser, set `DOWNLOAD_REPORT = True`, run the cell again and open the downloaded HTML file. Use this option if the report is blank in Colab.

**An older run:** replace `REPORT_OUTPUT = LAST_OUTPUT` with the batch's path from the Files panel, such as `ROOT / "outputs" / "tools-..."`.

You can also download `inspect.html` directly from the Files panel. Each run's report is inside its folder under `outputs/.../runs/`.


In [ ]:
REPORT_OUTPUT = LAST_OUTPUT
REPORT_RUN = 1
DOWNLOAD_REPORT = False

if REPORT_OUTPUT is None:
    print("Run the free demonstration or a campaign first, or set REPORT_OUTPUT to a saved batch folder.")
else:
    REPORT_FILE = show_report(REPORT_OUTPUT, REPORT_RUN)
    if DOWNLOAD_REPORT:
        download_file(REPORT_FILE)

### Check your spending

Run the next cell to see the estimated API spending recorded in this homework. This check is free. Run section 7's free helper cell first if you have not already.


In [ ]:
run_command("spend")  # Free: show the estimated spending recorded in this workspace.

## 10. Make the final runs and download your submission

The next cell makes **five runs** using the fixed course settings (about an hour and $5–8).

1. Save your final `prompt.py` and `tools.py`.
2. Set `RUN_FINAL = True` and run the cell. Keep the Colab session connected until all five runs finish.
3. Change `RUN_FINAL` back to `False`, then save a backup using section 11.

**If the session is interrupted:** restore your backup if needed, then run this cell again with the same files. Saved responses replay for free; missing responses incur normal API charges. The cell sets `FINAL_OUTPUT` only after all five runs complete.


In [ ]:
RUN_FINAL = False

if RUN_FINAL:
    prepare_paid()
    FINAL_OUTPUT = None
    output = default_output(ROOT, "final")
    run_command("final", "--output", output)
    results = [json.loads(p.read_text()) for p in sorted((output / "runs").glob("*/result.json"))]
    if len(results) != 5 or any(result.get("status") != "completed" for result in results):
        raise RuntimeError(f"The final batch did not complete five runs. Inspect {output} and rerun after fixing the issue.")
    FINAL_OUTPUT = output
    LAST_OUTPUT = output
    REPORT_FILE = show_report(FINAL_OUTPUT)
    print("Ready to package:", FINAL_OUTPUT)
else:
    print("Final batch skipped. Set RUN_FINAL = True when your files are ready.")

### Download your submission

1. After the five final runs finish, set `PACKAGE_SUBMISSION = True` and run the cell below. Packaging is free.
2. Submit the ZIP that downloads. The notebook and the backup ZIP are not your submission.

If you change `prompt.py` or `tools.py` after the final runs, repeat the final-run cell before packaging. Packaging checks that all five runs used the submitted files and required settings.

**After restoring a backup:** if `FINAL_OUTPUT` is not set, find the completed `final-...` folder under `outputs` in the Files panel. Set `FINAL_RUNS_FOLDER` below to `ROOT / "outputs" / "final-..."`, using that folder's exact name.


In [ ]:
PACKAGE_SUBMISSION = False
FINAL_RUNS_FOLDER = FINAL_OUTPUT  # After restoring: ROOT / "outputs" / "final-..."

if PACKAGE_SUBMISSION:
    if FINAL_RUNS_FOLDER is None:
        raise RuntimeError("Complete the final-run cell first, or select its saved final-... folder from the Files panel.")
    submission = default_output(ROOT, "submission", ".zip")
    run_command("package", FINAL_RUNS_FOLDER, "--output", submission)
    download_file(submission)
    print("Submit:", submission.name)
else:
    print("Packaging skipped. Set PACKAGE_SUBMISSION = True after the final runs complete.")

## 11. Save a backup before you leave

Colab can delete your working files when a session ends. Before closing it:

1. Save any changes to `prompt.py` and `tools.py` as shown in section 6.
2. Set `SAVE_BACKUP = True` and run the cell below. Keep the ZIP that downloads to your computer.
3. Save this notebook in Drive with **File → Save**, or download it with **File → Download → Download .ipynb**.

The ZIP saves your code, run results and saved model responses. Saving the notebook saves your edited cells. Do both: the ZIP does not capture edits to the notebook you have open.

Keeping the backup lets you replay previous runs for free and preserves your spending record. Your API key stays in Colab Secrets.

**Next time:** open your saved notebook and follow **Resume from a backup** in section 0. Before a paid run, follow section 7 to enable access to your Parley key.


In [ ]:
SAVE_BACKUP = False

if SAVE_BACKUP:
    import json
    import tempfile
    from datetime import datetime, timezone

    manifest = json.loads((ROOT / "handout_manifest.json").read_text())
    selected = {ROOT / name for name in manifest["files"]}
    selected.add(ROOT / "handout_manifest.json")
    for name in ("tests", "outputs", ".sales_cache"):
        directory = ROOT / name
        if directory.is_dir():
            selected.update(directory.rglob("*"))
    backup_dir = Path("/content") if IN_COLAB else Path(tempfile.gettempdir())
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
    backup = backup_dir / f"sales-agent-backup-{timestamp}.zip"
    skipped_names = {".venv", "venv", "__pycache__", ".git", ".ipynb_checkpoints", ".env"}
    with zipfile.ZipFile(backup, "x", compression=zipfile.ZIP_DEFLATED) as bundle:
        for file in sorted(selected):
            relative = file.relative_to(ROOT)
            if (not file.is_file() or file.is_symlink()
                    or any(part in skipped_names or part.startswith(".env.") for part in relative.parts)
                    or file.suffix.lower() in {".zip", ".key", ".pem", ".pyc"}):
                continue
            bundle.write(file, arcname=str(Path("sales-agent-homework") / relative))
    print("Backup saved:", backup)
    if IN_COLAB:
        from google.colab import files
        files.download(str(backup))
else:
    print("Backup skipped. Set SAVE_BACKUP = True before ending your session.")

## Optional: commands and files

The cells above cover the homework. Use this optional reference if you want to run an extra command.

1. Run section 0 first so the notebook uses the homework folder.
2. Add a code cell. Start the command with `!python run.py`, then add the command from the table below. For example, this checks your files for free:

```python
!python run.py test
```

**Before an extra paid command**, run section 7's helper cell, then run `prepare_paid()` in a code cell. It loads your Parley key and applies the spending cap. For example, this paid practice run costs a few cents:

```python
prepare_paid()
!python run.py run --case requirements_update
```

### Command reference

| `python run.py ...` | Cost | What it does |
| --- | --- | --- |
| `test` | free | Check your setup and your two files |
| `prompt` | free | Show exactly what the model reads |
| `run --case requirements_update --agent reference` | free | A scripted demo run, without a model |
| `run --case CASE` | cents | Your agent on a practice case: one scripted customer |
| `run --suite development/full_campaign/suite.json` | $1 | Your agent on the 40-account campaign |
| `final` | $5–8 | The five runs you hand in |
| `package OUTPUT` | free | The ZIP you hand in |
| `check OUTPUT` | small | A model's audit of saved runs: offers, answers, promises |
| `spend` | free | What your API calls have cost |

`--variant baseline` runs the supplied prompt and tools, `--variant prompt` your prompt with the supplied tools; the default is your prompt and tools. Each run saves a folder under `outputs/` with `inspect.html` and `transcript.md`. Running an unchanged run again replays it for free; add `--fresh` for new samples. The practice cases are `requirements_update`, `sourced_answer`, `issued_quote`, `callback_followthrough` and `optout`.

### Files

| File | |
| --- | --- |
| `prompt.py`, `tools.py` | Yours to change |
| `explore.ipynb` | Setup, free exercises, paid experiments and submission cells |
| `OPERATIONS.md` | What `ctx` can do, with real outputs |
| `tests/` | Offline checks run by `python run.py test`. Add tests for your tools. |
| `agent.py` | The loop that runs the agent. Read it; don't change it. |
| `run.py`, `course.json`, `practice/`, `development/` | Commands, settings and campaigns. Don't change them. |
